# Model to VRM: pipeline
One textured model in. Out: renders, keypoints, MHR body fit and stick skeleton, SAM 3 materials per vertex, contacts
labelled by the LLM (real contact or fold), and the flesh cut into islands. Every step shows its pictures below it.

**Run:** VS Code, kernel `E:\splat\.venv\Scripts\python.exe`, cells top to bottom. A step runs only when its outputs
are missing, so re-running is cheap; set `FORCE = True` in a step's cell to redo it. Steps that call the LLM read the
key from the vault and refuse to start if they would make more than `LLM_MAX_CALLS` calls.

In [ ]:
import os, sys, glob, subprocess, html, base64
from IPython.display import display, HTML
REPO = r'E:\code\model-to-vrm'
PY = r'E:\splat\.venv\Scripts\python.exe'; PYB = r'E:\splat\.venv-body\Scripts\python.exe'; PYS = r'E:\splat\.venv-sim\Scripts\python.exe'
BLENDER = r'C:\Program Files\Blender Foundation\Blender 5.3\blender.exe'
def sh(cmd, log=None):
    """run a command, print its last lines; stop the notebook on failure"""
    r = subprocess.run(cmd, capture_output=True, text=True, encoding='utf-8', errors='replace', cwd=REPO)
    out = (r.stdout or '') + (r.stderr or '')
    if log: open(log, 'a', encoding='utf-8').write(out)
    print('\n'.join(l for l in out.splitlines()[-15:] if 'it/s]' not in l))
    if r.returncode: raise RuntimeError('failed: %s' % ' '.join(cmd[:4]))
def show(*patterns, width=320):
    """show images side by side (glob patterns), newest render of each file"""
    files = [f for p in patterns for f in sorted(glob.glob(p))]
    if not files: print('no pictures for', patterns); return
    tags = []
    for f in files:
        ext = os.path.splitext(f)[1][1:].replace('jpg', 'jpeg'); b = base64.b64encode(open(f, 'rb').read()).decode()
        tags.append('<figure style="display:inline-block;margin:4px"><img src="data:image/%s;base64,%s" width="%d"><figcaption style="font-size:11px">%s</figcaption></figure>' % (ext, b, width, html.escape(os.path.basename(f))))
    display(HTML(''.join(tags)))

MODEL = r'E:\figures\model_r\test model R.glb'      # the input model (.glb, textured)
OUT = r'E:\figures\model_r'                   # everything for this model goes here
FACE = True                                 # also run the face-organ steps
os.environ['LLM_MAX_CALLS'] = '200'
def step(name, force=False):
    cmd = [PY, os.path.join(REPO, 'model_to_vrm', 'pipeline.py'), MODEL, OUT] + (['--face'] if FACE else []) + (['--only', name] if force else ['--step', name])
    sh(cmd)

## 1. Renders around the model (8 directions, textured and bare) and what each camera sees

In [ ]:
FORCE = False
step('views', FORCE); step('vis', FORCE)
show(OUT + r'\body\tex_a*.png', width=180); show(OUT + r'\body\clay_a*.png', width=180)

## 2. Per-vertex texture colour, normals, triangles

In [ ]:
FORCE = False
step('colours', FORCE); step('normals', FORCE)

## 3. Keypoints (RTMW on every view, triangulated) → MHR body fit → stick skeleton

In [ ]:
FORCE = False
step('joints', FORCE); show(OUT + r'\body\joints_check.jpg', width=900)
step('mhr', FORCE); step('skeleton', FORCE); step('sticks', FORCE)
show(OUT + r'\body\sticks_*.png', width=240)

## 4. SAM 3 masks → shares per vertex; then the vision LLM labels each segment (the model's own parts) skin / cloth / hair / prop

In [ ]:
FORCE = False
step('masks', FORCE); step('vote', FORCE); step('materials', FORCE); step('seglabel', FORCE)
print(open(OUT + r'\sam_body\materials.json').read())
show(OUT + r'\sam_body\seg_cards\seg_p[0-5].jpg', width=480)

## 5. Contacts: a ray along every normal → patches → split by material pair → a Blender card each

In [ ]:
FORCE = False
for s in ('rays', 'patches', 'combine', 'split', 'subrender', 'cards'): step(s, FORCE)
show(OUT + r'\contacts\sub_cards\sub_[0-5].jpg', width=480)

## 6. LLM per contact (parallel): real contact or a fold; its material flags → fixes

In [ ]:
FORCE = False
step('sublabel', FORCE); step('matfix', FORCE)
import json; L = json.load(open(OUT + r'\contacts\sub_labels.json'))
from collections import Counter; print(Counter(v.get('kind') for v in L.values())); print(open(OUT + r'\contacts\sub_fix.json').read())

## 7. Skin/cloth label per vertex: materials re-spread with the LLM fixes, specks and holes cleaned (used by the CT slice fit)

In [ ]:
FORCE = False
step('flesh', FORCE)
step('show_contacts', FORCE)
show(OUT + r'\contacts\sub_final\all_*.png', width=240)

## 8. Face organs from bare geometry (only with FACE = True)

In [ ]:
FORCE = False
if FACE:
    for s in ('geo_views', 'geo_masks', 'geo_vote', 'geo_paint'): step(s, FORCE)
    show(OUT + r'\geo\organs_*.png', width=320)